<a href="https://colab.research.google.com/github/adityab-tech/Prism/blob/main/Fine_Tuning_Kronos_Prism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers accelerate peft huggingface_hub safetensors

In [2]:
!git clone https://github.com/shiyu-coder/Kronos.git

Cloning into 'Kronos'...
remote: Enumerating objects: 371, done.
remote: Total 371 (delta 0), reused 0 (delta 0), pack-reused 371 (from 1)
Receiving objects: 100% (371/371), 9.35 MiB | 21.90 MiB/s, done.
Resolving deltas: 100% (158/158), done.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd Kronos

/content/Kronos


In [5]:
!ls

examples  finetune	LICENSE  README.md	   tests
figures   finetune_csv	model	 requirements.txt  webui


In [6]:
import os
print(os.getcwd())

/content/Kronos


In [7]:
!head -100 README.md

<div align="center">
  <h2><b>Kronos: A Foundation Model for the Language of Financial Markets </b></h2>
</div>


<div align="center">

</a> 
<a href="https://huggingface.co/NeoQuasar"> 
<img src="https://img.shields.io/badge/🤗-Hugging_Face-yellow" alt="Hugging Face"> 
</a> 
<a href="https://shiyu-coder.github.io/Kronos-demo/"> <img src="https://img.shields.io/badge/🚀-Live_Demo-brightgreen" alt="Live Demo"> </a>
<a href="https://github.com/shiyu-coder/Kronos/graphs/commit-activity"> 
<img src="https://img.shields.io/github/last-commit/shiyu-coder/Kronos?color=blue" alt="Last Commit"> 
</a> 
<a href="https://github.com/shiyu-coder/Kronos/stargazers"> 
<img src="https://img.shields.io/github/stars/shiyu-coder/Kronos?color=lightblue" alt="GitHub Stars"> 
</a> 
<a href="https://github.com/shiyu-coder/Kronos/network/members"> 
<img src="https://img.shields.io/github/forks/shiyu-coder/Kronos?color=yellow" alt="GitHub Forks"> 
</a> 
<a href="./LICENSE"> 
<img src="https://img.shields.io/githu

In [8]:
!ls model

__init__.py  kronos.py	module.py


In [9]:
!ls finetune

config.py   qlib_data_preprocess.py  train_predictor.py  utils
dataset.py  qlib_test.py	     train_tokenizer.py


In [10]:
!ls finetune_csv

config_loader.py  examples		  README_CN.md
configs		  finetune_base_model.py  README.md
data		  finetune_tokenizer.py   train_sequential.py


In [11]:
!sed -n '1,200p' finetune_csv/README.md

# Kronos Fine-tuning on Custom CSV Datasets

This module provides a comprehensive pipeline for fine-tuning Kronos models on your own CSV-formatted financial data. It supports both sequential training (tokenizer followed by predictor) and individual component training, with full distributed training capabilities.


## 1. Data Preparation

### Required Data Format

Your CSV file must contain the following columns:
- `timestamps`: DateTime stamps for each data point
- `open`: Opening price
- `high`: Highest price
- `low`: Lowest price  
- `close`: Closing price
- `volume`: Trading volume
- `amount`: Trading amount

(volume and amount can be 0 if not available)

### Sample Data Format

| timestamps | open | close | high | low | volume | amount |
|------------|------|-------|------|-----|--------|--------|
| 2019/11/26 9:35 | 182.45215 | 184.45215 | 184.95215 | 182.45215 | 15136000 | 0 |
| 2019/11/26 9:40 | 184.35215 | 183.85215 | 184.55215 | 183.45215 | 4433300 | 0 |
| 2019/11/26 9:45 | 18

In [12]:
!sed -n '1,250p' finetune_csv/finetune_base_model.py

import os
import sys
import json
import time
import pickle
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from time import gmtime, strftime
import logging
from logging.handlers import RotatingFileHandler
import datetime
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

sys.path.append('../')
from model import Kronos, KronosTokenizer, KronosPredictor
from config_loader import CustomFinetuneConfig


class CustomKlineDataset(Dataset):
    
    def __init__(self, data_path, data_type='train', lookback_window=90, predict_window=10, 
                 clip=5.0, seed=100, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
        self.data_path = data_path
        self.data_type = data_type
        self.lookback_window = lookback_window
        self.predict_window = predict_window
        self.window =

In [13]:
!sed -n '1,250p' finetune_csv/train_sequential.py

import os
import sys
import time
import argparse
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.distributed as dist

sys.path.append('../')
from model import Kronos, KronosTokenizer, KronosPredictor

from config_loader import CustomFinetuneConfig
from finetune_tokenizer import train_tokenizer, set_seed, setup_logging as setup_tokenizer_logging
from finetune_base_model import train_model, create_dataloaders, setup_logging as setup_basemodel_logging


class SequentialTrainer:
    
    def __init__(self, config_path: str = None):
        self.config = CustomFinetuneConfig(config_path)
        self.rank = int(os.environ.get("RANK", "0"))
        self.world_size = int(os.environ.get("WORLD_SIZE", "1"))
        self.local_rank = int(os.environ.get("LOCAL_RANK", str(self.config.device_id if hasattr(self.config, 'device_id') else 0)))
        self.device = self._setup_device()
        
        self.config.print_config_summary()
    
    def _setup_devic

In [14]:
!sed -n '1,250p' finetune_csv/config_loader.py

import os
import yaml
from typing import Dict, Any


class ConfigLoader:
    
    def __init__(self, config_path: str):

        self.config_path = config_path
        self.config = self._load_config()
    
    def _load_config(self) -> Dict[str, Any]:

        if not os.path.exists(self.config_path):
            raise FileNotFoundError(f"config file not found: {self.config_path}")
        
        with open(self.config_path, 'r', encoding='utf-8') as f:
            config = yaml.safe_load(f)
        
        config = self._resolve_dynamic_paths(config)
        
        return config
    
    def _resolve_dynamic_paths(self, config: Dict[str, Any]) -> Dict[str, Any]:

        exp_name = config.get('model_paths', {}).get('exp_name', '')
        if not exp_name:
            return config
        
        base_path = config.get('model_paths', {}).get('base_path', '')
        path_templates = {
            'base_save_path': f"{base_path}/{exp_name}",
            'finetuned_tokenizer': f"{b

In [15]:
import torch
import transformers
import huggingface_hub

print(torch.__version__)
print(transformers.__version__)
print(huggingface_hub.__version__)

2.11.0+cpu
5.12.1
1.20.1


In [16]:
!apt-get -qq install tree

Selecting previously unselected package tree.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...


In [17]:
!tree -L 2

.
├── examples
│   ├── get_akshare_date_2024-2025_x.py
│   ├── get_date_new.py
│   ├── prediction_akshare_2024-2025.py
│   ├── prediction_batch_example.py
│   ├── prediction_cn_markets_day.py
│   ├── prediction_example.py
│   ├── prediction_new_GUI.py
│   ├── prediction_new.py
│   ├── prediction_wo_vol_example.py
│   ├── run_backtest_kronos.py
│   └── yuce
├── figures
│   ├── backtest_result_example.png
│   ├── logo.png
│   ├── overview.png
│   └── prediction_example.png
├── finetune
│   ├── config.py
│   ├── dataset.py
│   ├── qlib_data_preprocess.py
│   ├── qlib_test.py
│   ├── train_predictor.py
│   ├── train_tokenizer.py
│   └── utils
├── finetune_csv
│   ├── config_loader.py
│   ├── configs
│   ├── data
│   ├── examples
│   ├── finetune_base_model.py
│   ├── finetune_tokenizer.py
│   ├── README_CN.md
│   ├── README.md
│   └── train_sequential.py
├── LICENSE
├── model
│   ├── __init__.py
│   ├── kronos.py
│   └── module.py
├── README.md
├── requirements.txt
├── tests
│   ├── data
│

In [18]:
important_folders = [
    "model",
    "finetune",
    "finetune_csv"
]

for folder in important_folders:
    print(folder)
    print(os.listdir(folder))

model
['kronos.py', '__init__.py', 'module.py']
finetune
['qlib_test.py', 'train_tokenizer.py', 'config.py', 'utils', 'qlib_data_preprocess.py', 'train_predictor.py', 'dataset.py']
finetune_csv
['train_sequential.py', 'finetune_base_model.py', 'examples', 'finetune_tokenizer.py', 'README.md', 'config_loader.py', 'data', 'README_CN.md', 'configs']


In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [20]:
#Repository Structure

## model/
#- Contains the core Kronos model architecture.
#- Includes the Transformer blocks, tokenizer implementation, and model modules.
#- This is where the pretrained Kronos model is defined.

## finetune/
#- Contains the original fine-tuning pipeline provided by the authors.
#- Used for training the tokenizer and predictor separately.

## finetune_csv/
#- Contains the CSV-based fine-tuning pipeline.
#- This is the folder we will use for the PRISM project because our stock data is stored in CSV files.
#- Includes training scripts, configuration files, and dataset loading utilities.

## configs/
#- Stores YAML configuration files.
#- Defines parameters such as lookback window, prediction window, batch size, learning rate, and data paths.

## data/
#- Stores the input CSV files used for training.
#- For our project, we will replace these with Indian stock CSVs.

## train_sequential.py
#- Main script that performs sequential training.
#- It trains the tokenizer first and then the Kronos predictor.

In [21]:
!ls finetune_csv/configs

config_ali09988_candle-5min.yaml


In [22]:
!sed -n '1,200p' finetune_csv/configs/config_ali09988_candle-5min.yaml

#This is a template config for custom finetuning kronos on csv data
#这是一份模板config，用于kronos的csv自定义数据微调

data:
  data_path: "/xxxx/Kronos/finetune_csv/data/HK_ali_09988_kline_5min_all.csv"
  lookback_window: 512
  predict_window: 48
  max_context: 512
  clip: 5.0
  # dataset split ratio
  train_ratio: 0.9
  val_ratio: 0.1
  test_ratio: 0.0

training:
  # control the training epochs of tokenizer and basemodel
  tokenizer_epochs: 30
  basemodel_epochs: 20
  batch_size: 32
  log_interval: 50
  num_workers: 6
  seed: 42
  
  tokenizer_learning_rate: 0.0002
  predictor_learning_rate: 0.000001
  
  adam_beta1: 0.9
  adam_beta2: 0.95
  adam_weight_decay: 0.1
  
  # gradient accumulation steps for tokenizer training
  accumulation_steps: 1

# model path configuration
model_paths:
  # pretrained model path
  pretrained_tokenizer: "/xxx/Kronos/pretrained/Kronos-Tokenizer-base"
  pretrained_predictor: "/xxx/Kronos/pretrained/Kronos-base"
  
  # experiment name - other paths will be generated based 

In [23]:
from finetune_csv.config_loader import CustomFinetuneConfig

In [24]:
config = CustomFinetuneConfig(
    "finetune_csv/configs/config_ali09988_candle-5min.yaml")

In [25]:
config.print_config_summary()

Kronos finetuning configuration summary
Experiment name: HK_ali_09988_kline_5min_all
Data path: /xxxx/Kronos/finetune_csv/data/HK_ali_09988_kline_5min_all.csv
Lookback window: 512
Predict window: 48
Tokenizer training epochs: 30
Basemodel training epochs: 20
Batch size: 32
Tokenizer learning rate: 0.0002
Predictor learning rate: 1e-06
Train tokenizer: True
Train basemodel: True
Skip existing: False
Use pre-trained tokenizer: True
Use pre-trained predictor: True
Base save path: /xxx/Kronos/finetune_csv/finetuned//HK_ali_09988_kline_5min_all
Tokenizer save path: /xxx/Kronos/finetune_csv/finetuned//HK_ali_09988_kline_5min_all/tokenizer
Basemodel save path: /xxx/Kronos/finetune_csv/finetuned//HK_ali_09988_kline_5min_all/basemodel


In [26]:
#upar jo output mai given hai bass usse wapas print karwa raha hu taaki easy dikhe lol nigga
print("Data Path :", config.data_path)
print("Lookback :", config.lookback_window)
print("Predict :", config.predict_window)
print("Batch Size :", config.batch_size)
print("Tokenizer LR :", config.tokenizer_learning_rate)
print("Predictor LR :", config.predictor_learning_rate)

Data Path : /xxxx/Kronos/finetune_csv/data/HK_ali_09988_kline_5min_all.csv
Lookback : 512
Predict : 48
Batch Size : 32
Tokenizer LR : 0.0002
Predictor LR : 1e-06


In [27]:
#lookback_window -> Number of past candles.
#predict_window-> Future candles.
#batch_size -> Training batch.
#pretrained_predictor_path->Kronos-base.
#pretrained_tokenizer_path->Tokenizer.

In [28]:
!sed -n '1,250p' finetune/dataset.py

import pickle
import random
import numpy as np
import torch
from torch.utils.data import Dataset
from config import Config


class QlibDataset(Dataset):
    """
    A PyTorch Dataset for handling Qlib financial time series data.

    This dataset pre-computes all possible start indices for sliding windows
    and then randomly samples from them during training/validation.

    Args:
        data_type (str): The type of dataset to load, either 'train' or 'val'.

    Raises:
        ValueError: If `data_type` is not 'train' or 'val'.
    """

    def __init__(self, data_type: str = 'train'):
        self.config = Config()
        if data_type not in ['train', 'val']:
            raise ValueError("data_type must be 'train' or 'val'")
        self.data_type = data_type

        # Use a dedicated random number generator for sampling to avoid
        # interfering with other random processes (e.g., in model initialization).
        self.py_rng = random.Random(self.config.seed)

        # Set

In [29]:
!grep -n "class" finetune/dataset.py

9:class QlibDataset(Dataset):


In [30]:
!grep -n "__getitem__" finetune/dataset.py
!grep -n "__len__" finetune/dataset.py

92:    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
88:    def __len__(self) -> int:


In [31]:
#CSV ->Read using pandas-> Sliding Window ->Lookback Window ->Predict Window ->Dataset ->DataLoader

In [32]:
import os
data_path = "finetune_csv/data"
print(os.listdir(data_path))

['HK_ali_09988_kline_5min_all.csv']


In [33]:
sample_df = pd.read_csv(
    "finetune_csv/data/HK_ali_09988_kline_5min_all.csv")
sample_df.head()

,timestamps,open,close,high,low,volume,amount
0,2019/11/26 9:35,182.45215,184.45215,184.95215,182.45215,15136000,0
1,2019/11/26 9:40,184.35215,183.85215,184.55215,183.45215,4433300,0
2,2019/11/26 9:45,183.85215,183.35215,183.95215,182.95215,3070900,0
3,2019/11/26 9:50,183.35215,183.75215,183.95215,183.25215,2288600,0
4,2019/11/26 9:55,183.85215,183.85215,184.15215,183.75215,1770000,0


In [34]:
print(sample_df.columns.tolist())

['timestamps', 'open', 'close', 'high', 'low', 'volume', 'amount']


In [35]:
sample_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93912 entries, 0 to 93911
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   timestamps  93912 non-null  object 
 1   open        93912 non-null  float64
 2   close       93912 non-null  float64
 3   high        93912 non-null  float64
 4   low         93912 non-null  float64
 5   volume      93912 non-null  int64  
 6   amount      93912 non-null  int64  
dtypes: float64(4), int64(2), object(1)
memory usage: 5.0+ MB


In [36]:
my_df = pd.read_csv(
    "/content/drive/MyDrive/PRISM/raw_data/TCS.NS.csv"
)
print("My Dataset Columns:")
print(my_df.columns.tolist())

print("\nKronos Dataset Columns:")
print(sample_df.columns.tolist())

My Dataset Columns:
['Date', 'Close', 'High', 'Low', 'Open', 'Volume']

Kronos Dataset Columns:
['timestamps', 'open', 'close', 'high', 'low', 'volume', 'amount']


In [37]:
sample_df["timestamps"] = pd.to_datetime(sample_df["timestamps"])
my_df["Date"] = pd.to_datetime(my_df["Date"])

In [38]:
!sed -n '1,200p' finetune_csv/finetune_base_model.py

import os
import sys
import json
import time
import pickle
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from time import gmtime, strftime
import logging
from logging.handlers import RotatingFileHandler
import datetime
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

sys.path.append('../')
from model import Kronos, KronosTokenizer, KronosPredictor
from config_loader import CustomFinetuneConfig


class CustomKlineDataset(Dataset):
    
    def __init__(self, data_path, data_type='train', lookback_window=90, predict_window=10, 
                 clip=5.0, seed=100, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
        self.data_path = data_path
        self.data_type = data_type
        self.lookback_window = lookback_window
        self.predict_window = predict_window
        self.window =

In [39]:
!sed -n '1,200p' finetune_csv/train_sequential.py

import os
import sys
import time
import argparse
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.distributed as dist

sys.path.append('../')
from model import Kronos, KronosTokenizer, KronosPredictor

from config_loader import CustomFinetuneConfig
from finetune_tokenizer import train_tokenizer, set_seed, setup_logging as setup_tokenizer_logging
from finetune_base_model import train_model, create_dataloaders, setup_logging as setup_basemodel_logging


class SequentialTrainer:
    
    def __init__(self, config_path: str = None):
        self.config = CustomFinetuneConfig(config_path)
        self.rank = int(os.environ.get("RANK", "0"))
        self.world_size = int(os.environ.get("WORLD_SIZE", "1"))
        self.local_rank = int(os.environ.get("LOCAL_RANK", str(self.config.device_id if hasattr(self.config, 'device_id') else 0)))
        self.device = self._setup_device()
        
        self.config.print_config_summary()
    
    def _setup_devic

In [40]:
!sed -n '1,200p' finetune_csv/config_loader.py

import os
import yaml
from typing import Dict, Any


class ConfigLoader:
    
    def __init__(self, config_path: str):

        self.config_path = config_path
        self.config = self._load_config()
    
    def _load_config(self) -> Dict[str, Any]:

        if not os.path.exists(self.config_path):
            raise FileNotFoundError(f"config file not found: {self.config_path}")
        
        with open(self.config_path, 'r', encoding='utf-8') as f:
            config = yaml.safe_load(f)
        
        config = self._resolve_dynamic_paths(config)
        
        return config
    
    def _resolve_dynamic_paths(self, config: Dict[str, Any]) -> Dict[str, Any]:

        exp_name = config.get('model_paths', {}).get('exp_name', '')
        if not exp_name:
            return config
        
        base_path = config.get('model_paths', {}).get('base_path', '')
        path_templates = {
            'base_save_path': f"{base_path}/{exp_name}",
            'finetuned_tokenizer': f"{b

In [41]:
!sed -n '1,200p' model/kronos.py

import numpy as np
import pandas as pd
import torch
from huggingface_hub import PyTorchModelHubMixin
import sys

from tqdm import trange

sys.path.append("../")
from model.module import *


class KronosTokenizer(nn.Module, PyTorchModelHubMixin):
    """
    KronosTokenizer module for tokenizing input data using a hybrid quantization approach.

    This tokenizer utilizes a combination of encoder and decoder Transformer blocks
    along with the Binary Spherical Quantization (BSQuantizer) to compress and decompress input data.

    Args:
           d_in (int): Input dimension.
           d_model (int): Model dimension.
           n_heads (int): Number of attention heads.
           ff_dim (int): Feed-forward dimension.
           n_enc_layers (int): Number of encoder layers.
           n_dec_layers (int): Number of decoder layers.
           ffn_dropout_p (float): Dropout probability for feed-forward networks.
           attn_dropout_p (float): Dropout probability for attention mechanis

In [42]:
import os
import sys

os.chdir("/content/Kronos")

sys.path.append("/content/Kronos")
sys.path.append("/content/Kronos/finetune_csv")
sys.path.append("/content/Kronos/model")

print(os.getcwd())

/content/Kronos


In [43]:
from finetune_csv.config_loader import CustomFinetuneConfig

from finetune_csv.finetune_base_model import (
    create_dataloaders,
    CustomKlineDataset
)

from model import Kronos, KronosTokenizer, KronosPredictor

In [44]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [45]:
config = CustomFinetuneConfig(
    "finetune_csv/configs/config_ali09988_candle-5min.yaml")

In [46]:
print("Data Path :", config.data_path)
print("Lookback :", config.lookback_window)
print("Predict :", config.predict_window)
print("Batch Size :", config.batch_size)
print("Tokenizer LR :", config.tokenizer_learning_rate)
print("Predictor LR :", config.predictor_learning_rate)

Data Path : /xxxx/Kronos/finetune_csv/data/HK_ali_09988_kline_5min_all.csv
Lookback : 512
Predict : 48
Batch Size : 32
Tokenizer LR : 0.0002
Predictor LR : 1e-06


In [47]:
config = CustomFinetuneConfig(
    "/content/Kronos/finetune_csv/configs/config_ali09988_candle-5min.yaml")
config.data_path = "/content/Kronos/finetune_csv/data/HK_ali_09988_kline_5min_all.csv"

In [48]:
train_dataset = CustomKlineDataset(
    data_path=config.data_path,
    data_type="train",
    lookback_window=config.lookback_window,
    predict_window=config.predict_window
)

Original data time range: 2019-11-26 09:35:00 to 2025-09-17 16:00:00
Original data total length: 93912 records
[TRAIN] Training set: first 65738 time points (0.7)
[TRAIN] Training set time range: 2019-11-26 09:35:00 to 2023-12-18 15:40:00
[TRAIN] Data length after split: 65738 records
[TRAIN] Data length: 65738, Available samples: 65178


In [49]:
print(len(train_dataset))

65178


In [50]:
x, x_stamp = train_dataset[0]
print("Input Shape :", x.shape)
print("Time Shape :", x_stamp.shape)

Input Shape : torch.Size([561, 6])
Time Shape : torch.Size([561, 5])


In [51]:
!grep -n "return" /content/Kronos/finetune_csv/finetune_base_model.py

105:        return self.n_samples
132:        return x_tensor, x_stamp_tensor
144:        return logger
178:    return logger
236:    return train_loader, val_loader, train_dataset, val_dataset, train_sampler, val_sampler
364:    return best_val_loss


In [52]:
print(config.data_path)

/content/Kronos/finetune_csv/data/HK_ali_09988_kline_5min_all.csv


In [53]:
!find /content/Kronos -name "*.csv"

/content/Kronos/tests/data/regression_output_512.csv
/content/Kronos/tests/data/regression_input.csv
/content/Kronos/tests/data/regression_output_256.csv
/content/Kronos/finetune_csv/data/HK_ali_09988_kline_5min_all.csv


In [54]:
config.data_path = "/content/Kronos/finetune_csv/data/HK_ali_09988_kline_5min_all.csv"
print(config.data_path)

/content/Kronos/finetune_csv/data/HK_ali_09988_kline_5min_all.csv


In [55]:
train_loader, val_loader, train_dataset, val_dataset, train_sampler, val_sampler = create_dataloaders(config)

Creating data loaders...
Original data time range: 2019-11-26 09:35:00 to 2025-09-17 16:00:00
Original data total length: 93912 records
[TRAIN] Training set: first 84520 time points (0.9)
[TRAIN] Training set time range: 2019-11-26 09:35:00 to 2025-02-21 14:20:00
[TRAIN] Data length after split: 84520 records
[TRAIN] Data length: 84520, Available samples: 83960
Original data time range: 2019-11-26 09:35:00 to 2025-09-17 16:00:00
Original data total length: 93912 records
[VAL] Validation set: time points 84521 to 93912 (0.1)
[VAL] Validation set time range: 2025-02-21 14:25:00 to 2025-09-17 16:00:00
[VAL] Data length after split: 9392 records
[VAL] Data length: 9392, Available samples: 8832
Training set size: 83960, Validation set size: 8832


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [56]:
for x, x_stamp in train_loader:
    print("Input Shape :", x.shape)
    print("Timestamp Shape :", x_stamp.shape)
    break

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Input Shape : torch.Size([32, 561, 6])
Timestamp Shape : torch.Size([32, 561, 5])


In [57]:
print("Maximum Value :", x.max())
print("Minimum Value :", x.min())
print("Mean :", x.mean())
print("Std :", x.std())

Maximum Value : tensor(5.)
Minimum Value : tensor(-4.1070)
Mean : tensor(-0.0036)
Std : tensor(0.8814)


In [58]:
print(x_stamp[0])

tensor([[20., 13.,  4.,  4., 11.],
        [25., 13.,  4.,  4., 11.],
        [30., 13.,  4.,  4., 11.],
        ...,
        [50., 15.,  2., 16., 11.],
        [55., 15.,  2., 16., 11.],
        [ 0., 16.,  2., 16., 11.]])


In [59]:
print(config.pretrained_tokenizer_path)
print(config.pretrained_predictor_path)

/xxx/Kronos/pretrained/Kronos-Tokenizer-base
/xxx/Kronos/pretrained/Kronos-base


In [60]:
!find /content/Kronos -type f -name "config.json"

In [61]:
!find /content/Kronos -type d | grep pretrained

In [62]:
!grep -R "from_pretrained" -n /content/Kronos

/content/Kronos/finetune/qlib_test.py:211:    tokenizer = KronosTokenizer.from_pretrained(config['tokenizer_path']).to(device).eval()
/content/Kronos/finetune/qlib_test.py:212:    model = Kronos.from_pretrained(config['model_path']).to(device).eval()
/content/Kronos/finetune/train_tokenizer.py:251:    model = KronosTokenizer.from_pretrained(config['pretrained_tokenizer_path'])
/content/Kronos/finetune/train_predictor.py:213:    tokenizer = KronosTokenizer.from_pretrained(config['finetuned_tokenizer_path'])
/content/Kronos/finetune/train_predictor.py:216:    model = Kronos.from_pretrained(config['pretrained_predictor_path'])
/content/Kronos/webui/app.py:645:        tokenizer = KronosTokenizer.from_pretrained(model_config['tokenizer_id'])
/content/Kronos/webui/app.py:646:        model = Kronos.from_pretrained(model_config['model_id'])
/content/Kronos/examples/prediction_akshare_2024-2025.py:401:        tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
/conten

In [63]:
!find /content/Kronos -name "*.pt" -o -name "*.bin" -o -name "*.safetensors"

In [64]:
config.pretrained_tokenizer_path = "NeoQuasar/Kronos-Tokenizer-base"
config.pretrained_predictor_path = "NeoQuasar/Kronos-base"

In [65]:
tokenizer = KronosTokenizer.from_pretrained(
    config.pretrained_tokenizer_path
).to(device)

print("Tokenizer Loaded!")

config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

Tokenizer Loaded!


In [66]:
predictor = Kronos.from_pretrained(
    config.pretrained_predictor_path
).to(device)
print("Predictor Loaded!")

config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/409M [00:00<?, ?B/s]

Predictor Loaded!


In [67]:
total = sum(
    p.numel()
    for p in tokenizer.parameters())
print(f"Total Parameters : {total:,}")

Total Parameters : 3,958,042


In [68]:
print(tokenizer)

KronosTokenizer(
  (embed): Linear(in_features=6, out_features=256, bias=True)
  (head): Linear(in_features=256, out_features=6, bias=True)
  (encoder): ModuleList(
    (0-2): 3 x TransformerBlock(
      (norm1): RMSNorm()
      (self_attn): MultiHeadAttentionWithRoPE(
        (q_proj): Linear(in_features=256, out_features=256, bias=True)
        (k_proj): Linear(in_features=256, out_features=256, bias=True)
        (v_proj): Linear(in_features=256, out_features=256, bias=True)
        (out_proj): Linear(in_features=256, out_features=256, bias=True)
        (rotary): RotaryPositionalEmbedding()
        (resid_dropout): Dropout(p=0.0, inplace=False)
      )
      (norm2): RMSNorm()
      (ffn): FeedForward(
        (w1): Linear(in_features=256, out_features=512, bias=False)
        (w3): Linear(in_features=256, out_features=512, bias=False)
        (w2): Linear(in_features=512, out_features=256, bias=False)
        (ffn_dropout): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (decode

In [69]:
for name, module in tokenizer.named_children():
    print(name)

embed
head
encoder
decoder
quant_embed
post_quant_embed_pre
post_quant_embed
tokenizer


In [70]:
for name, param in tokenizer.named_parameters():
    print(
        name,
        param.shape,
        param.requires_grad
    )

embed.weight torch.Size([256, 6]) True
embed.bias torch.Size([256]) True
head.weight torch.Size([6, 256]) True
head.bias torch.Size([6]) True
encoder.0.norm1.weight torch.Size([256]) True
encoder.0.self_attn.q_proj.weight torch.Size([256, 256]) True
encoder.0.self_attn.q_proj.bias torch.Size([256]) True
encoder.0.self_attn.k_proj.weight torch.Size([256, 256]) True
encoder.0.self_attn.k_proj.bias torch.Size([256]) True
encoder.0.self_attn.v_proj.weight torch.Size([256, 256]) True
encoder.0.self_attn.v_proj.bias torch.Size([256]) True
encoder.0.self_attn.out_proj.weight torch.Size([256, 256]) True
encoder.0.self_attn.out_proj.bias torch.Size([256]) True
encoder.0.norm2.weight torch.Size([256]) True
encoder.0.ffn.w1.weight torch.Size([512, 256]) True
encoder.0.ffn.w3.weight torch.Size([512, 256]) True
encoder.0.ffn.w2.weight torch.Size([256, 512]) True
encoder.1.norm1.weight torch.Size([256]) True
encoder.1.self_attn.q_proj.weight torch.Size([256, 256]) True
encoder.1.self_attn.q_proj.bia

In [71]:
from model import Kronos
config.pretrained_predictor_path = "NeoQuasar/Kronos-base"
model = Kronos.from_pretrained(
    config.pretrained_predictor_path)
model = model.to(device)
print("Kronos Predictor Loaded Successfully!")

Kronos Predictor Loaded Successfully!


In [72]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad)
print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

Total Parameters     : 102,310,592
Trainable Parameters : 102,310,592


In [73]:
print(model)

Kronos(
  (token_drop): Dropout(p=0.0, inplace=False)
  (embedding): HierarchicalEmbedding(
    (emb_s1): Embedding(1024, 832)
    (emb_s2): Embedding(1024, 832)
    (fusion_proj): Linear(in_features=1664, out_features=832, bias=True)
  )
  (time_emb): TemporalEmbedding(
    (minute_embed): Embedding(60, 832)
    (hour_embed): Embedding(24, 832)
    (weekday_embed): Embedding(7, 832)
    (day_embed): Embedding(32, 832)
    (month_embed): Embedding(13, 832)
  )
  (transformer): ModuleList(
    (0-11): 12 x TransformerBlock(
      (norm1): RMSNorm()
      (self_attn): MultiHeadAttentionWithRoPE(
        (q_proj): Linear(in_features=832, out_features=832, bias=True)
        (k_proj): Linear(in_features=832, out_features=832, bias=True)
        (v_proj): Linear(in_features=832, out_features=832, bias=True)
        (out_proj): Linear(in_features=832, out_features=832, bias=True)
        (rotary): RotaryPositionalEmbedding()
        (resid_dropout): Dropout(p=0.2, inplace=False)
      )
    

In [74]:
for name, module in model.named_children():
    print(name)

token_drop
embedding
time_emb
transformer
norm
dep_layer
head


In [75]:
for name, param in model.named_parameters():
    print(
        name,
        param.shape,
        param.requires_grad
    )

embedding.emb_s1.weight torch.Size([1024, 832]) True
embedding.emb_s2.weight torch.Size([1024, 832]) True
embedding.fusion_proj.weight torch.Size([832, 1664]) True
embedding.fusion_proj.bias torch.Size([832]) True
time_emb.minute_embed.weight torch.Size([60, 832]) True
time_emb.hour_embed.weight torch.Size([24, 832]) True
time_emb.weekday_embed.weight torch.Size([7, 832]) True
time_emb.day_embed.weight torch.Size([32, 832]) True
time_emb.month_embed.weight torch.Size([13, 832]) True
transformer.0.norm1.weight torch.Size([832]) True
transformer.0.self_attn.q_proj.weight torch.Size([832, 832]) True
transformer.0.self_attn.q_proj.bias torch.Size([832]) True
transformer.0.self_attn.k_proj.weight torch.Size([832, 832]) True
transformer.0.self_attn.k_proj.bias torch.Size([832]) True
transformer.0.self_attn.v_proj.weight torch.Size([832, 832]) True
transformer.0.self_attn.v_proj.bias torch.Size([832]) True
transformer.0.self_attn.out_proj.weight torch.Size([832, 832]) True
transformer.0.self_

In [76]:
print("Configuration Summary")
print("Lookback Window :", config.lookback_window)
print("Prediction Window :", config.predict_window)
print("Batch Size :", config.batch_size)

print("Tokenizer LR :", config.tokenizer_learning_rate)
print("Predictor LR :", config.predictor_learning_rate)

Configuration Summary
Lookback Window : 512
Prediction Window : 48
Batch Size : 32
Tokenizer LR : 0.0002
Predictor LR : 1e-06


In [77]:
print("Train Ratio :", config.train_ratio)
print("Validation Ratio :", config.val_ratio)
print("Test Ratio :", config.test_ratio)
print("Clip Value :", config.clip)

Train Ratio : 0.9
Validation Ratio : 0.1
Test Ratio : 0.0
Clip Value : 5.0


In [78]:
print("Dataset Path")
print(config.data_path)
print("Tokenizer")
print(config.pretrained_tokenizer_path)
print("Predictor")
print(config.pretrained_predictor_path)

Dataset Path
/content/Kronos/finetune_csv/data/HK_ali_09988_kline_5min_all.csv
Tokenizer
NeoQuasar/Kronos-Tokenizer-base
Predictor
NeoQuasar/Kronos-base


In [79]:
print("Experiment Name")
print(config.exp_name)

Experiment Name
HK_ali_09988_kline_5min_all


In [80]:
print(f"Tokenizer Parameters : {sum(p.numel() for p in tokenizer.parameters()):,}")
print(f"Kronos Parameters : {sum(p.numel() for p in model.parameters()):,}")

Tokenizer Parameters : 3,958,042
Kronos Parameters : 102,310,592


In [81]:
print("First Batch Shape")
for x, x_stamp in train_loader:
    print(x.shape)
    print(x_stamp.shape)
    break

First Batch Shape
torch.Size([32, 561, 6])
torch.Size([32, 561, 5])


In [82]:
print("Sample Input")
print(x[0])

Sample Input
tensor([[ 2.2987,  2.1922,  2.2181,  2.1309, -0.1065,  0.0000],
        [ 2.1428,  2.1005,  2.2181,  2.1622, -0.1653,  0.0000],
        [ 2.1428,  2.1005,  2.2499,  2.2249,  0.0622,  0.0000],
        ...,
        [-1.3809, -1.3692, -1.3744, -1.4111, -0.5876,  0.0000],
        [-1.4277, -1.4456, -1.4221, -1.4267, -0.5790,  0.0000],
        [-1.4277, -1.4303, -1.4221, -1.4738, -0.7041,  0.0000]])


In [83]:
print("Sample Time Features")
print(x_stamp[0])

Sample Time Features
tensor([[45., 14.,  1.,  8., 10.],
        [50., 14.,  1.,  8., 10.],
        [55., 14.,  1.,  8., 10.],
        ...,
        [45., 10.,  1., 22., 10.],
        [50., 10.,  1., 22., 10.],
        [55., 10.,  1., 22., 10.]])
